<a href="https://colab.research.google.com/github/tharunika-19/RTR-project-ML-part-/blob/main/earthquake_model_pkl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

In [ ]:
df = pd.read_csv("enhanced_earthquake_data.csv")
print(df.columns)
print(df.head())
print(df.shape)
print(df.isnull().sum())

Index(['id', 'place', 'time', 'updated', 'magnitude', 'mag_type', 'alert',
       'status', 'tsunami', 'significance', 'felt', 'cdi', 'mmi', 'nst', 'gap',
       'rms', 'longitude', 'latitude', 'depth', 'url', 'detail', 'day_of_week',
       'hour'],
      dtype='object')
             id                                      place  \
0    ci40193954                  4 km ESE of San Diego, CA   
1  ak0252kuqzlf               26 km NNW of Susitna, Alaska   
2    ok2025dxpl         3 km WNW of Bridge Creek, Oklahoma   
3    nc75139036                 7 km NW of The Geysers, CA   
4  ak0252kuleb5  57 km SSE of Denali National Park, Alaska   

                      time                  updated  magnitude mag_type alert  \
0  2025-02-25 14:54:03.680  2025-02-25 14:57:37.312       1.29       ml   NaN   
1  2025-02-25 14:53:58.591  2025-02-25 14:56:06.492       1.10       ml   NaN   
2  2025-02-25 14:32:11.500  2025-02-25 14:50:24.973      -0.19       ml   NaN   
3  2025-02-25 14:30:59.700  20

In [ ]:
df = df[['magnitude', 'depth', 'tsunami', 'significance']]
print(df.head())
print(df.isnull().sum())

   magnitude  depth  tsunami  significance
0       1.29  23.74        0            26
1       1.10  50.60        0            19
2      -0.19   5.76        0             0
3       0.74   2.42        0             8
4       1.40   1.00        0            30
magnitude       0
depth           0
tsunami         0
significance    0
dtype: int64


In [ ]:
df = df.dropna()
print(df.shape)
print(df.isnull().sum())

(1958, 4)
magnitude       0
depth           0
tsunami         0
significance    0
dtype: int64


In [ ]:
def assign_risk(magnitude):
    if magnitude >= 5.0:
        return 'HIGH'
    elif magnitude >= 3.0:
        return 'MEDIUM'
    else:
        return 'LOW'

df['risk_level'] = df['magnitude'].apply(assign_risk)
print(df['risk_level'].value_counts())

risk_level
LOW       1761
MEDIUM     166
HIGH        31
Name: count, dtype: int64


In [ ]:
df['risk_level'] = df['risk_level'].map({'LOW': 0, 'MEDIUM': 1, 'HIGH': 2})
print(df.head())

   magnitude  depth  tsunami  significance  risk_level
0       1.29  23.74        0            26           0
1       1.10  50.60        0            19           0
2      -0.19   5.76        0             0           0
3       0.74   2.42        0             8           0
4       1.40   1.00        0            30           0


In [ ]:
X = df.drop('risk_level', axis=1)
y = df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 1566
Testing rows: 392


In [ ]:
model = DecisionTreeClassifier()
model.fit(X_train, y_train)
print("Model trained successfully!")

Model trained successfully!


In [ ]:
predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       357
           1       1.00      1.00      1.00        30
           2       1.00      1.00      1.00         5

    accuracy                           1.00       392
   macro avg       1.00      1.00      1.00       392
weighted avg       1.00      1.00      1.00       392

Confusion Matrix:
[[357   0   0]
 [  0  30   0]
 [  0   0   5]]


In [ ]:
with open('earthquake_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved successfully!")

Model saved successfully!


In [ ]:
with open('earthquake_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

sample = pd.DataFrame([[0, 0, 0, 0]],
                       columns=['magnitude', 'depth', 'tsunami', 'significance'])

result = loaded_model.predict(sample)
risk = {0: 'LOW', 1: 'MEDIUM', 2: 'HIGH'}
print("Earthquake Risk Level:", risk[result[0]])

Earthquake Risk Level: LOW
